***

# **BLS Queries**

***

In this file, we want to try and pull the total number of employment from BLS for different industries. To give some context, the API works by specifying a specific series id, which can be used to pull data from a specific table from BLS. It can also be used to select information from a table, parameters like specific counties or specific industry can be referenced. The survey we are trying to pull from is State and Employment, Hours, and Earnings; which can be found in the following link: https://www.bls.gov/help/hlpforma.htm#EW. 

***

## Prepare Workspace

***

In [ ]:
# Packages
import pandas as pd
import json
import requests
import os

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')



print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# Set API key
exec(open(os.path.join(path_config, 'api_key.txt')).read())
key = dict_api[user]
key = '?registrationkey={}'.format(key)

***

## User-Defined Functions

***

In [ ]:
# Create a function to make all of these counties into a dictionary

def dict_maker(df, sector, pre, data_type):
    """
    Given the file: BLS Configuration File.xlsx under the BLS_MSA sheet, we can create a dictionary of 
    all of the MSA counties we want to test. Provide the sector (industry) that you want to pull, and the function will
    return a dictionary of all the MSA series ids formatted for API usage. We must define the first series ID manually,
    but the rest is automated (probably a better way to do it).
    """

    # Formula for Series ID = Prefix + SA + State + Area + Industry + DType
    # pre = "SMU"    
    # data_type = '01'

    # Making set of keys and vals for future dict
    keys = []

    # Now empty list for values in the future dict
    vals = []

    for i in range(len(df)):

        # Getting each code
        area_code = str(df.iloc[i, 0])
        state     = str(df.iloc[i, 2])

        # Making each SeriesID
        series_id = pre + state + area_code + sector + data_type

        # Adding to the keylist for future dictionary
        keys.append(series_id)

        val = str(df.iloc[i, 1])

        vals.append(val)

    result = {k: v for k, v in zip(keys, vals)}

    return result
        

In [ ]:
# Need to update bls_query() so that it doesn't only do the first 10 years

# Let's update bls_query() so that we can make it so that each series is uniform and has 120 rows for all

def bls_query_update(series_dict, dates, api_key):
    """ 
    This function takes a dictionary of series, and a series of dates in the form dates = (start, year) to return
    a dataframe with information regarding employment in the sector that the user prescribes. Because of BLS's 
    query limit of up to 10 years of data being pulled at a time for each series ID, the function loops over a set of 10 or less.
    Meaning that if you supply it with years 2000-2024, it will loop three times subsetting between 2000-2009, 2010-2019, 2020-2024.
    It can also work in year ranges less than 10, so if you want to just pull say 2020-2024, that is totally viable.  
    """

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    key = '?registrationkey={}'.format(api_key)

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Initialize an empty dataframe to store our query results
    list_df = []

    # Queries ten years at once
    year_step = 10

    # Loop through the specified range of years in step intervals
    for year_range_start in range(dates[0], dates[1] + 1, year_step):
        year_range_end = min(year_range_start + year_step - 1, dates[1])

        df = pd.DataFrame()

        # Used a print statement for troubleshooting
        # print("Querying data for years {}-{}".format(year_range_start, year_range_end))

        # Submit the request for the current date range
        data = json.dumps({
            "seriesid": list(series_dict.keys()),
            "startyear": year_range_start,
            "endyear": year_range_end})
        response = requests.post('{}{}'.format(url, key), headers=headers, data=data).json()

        # Extract data from the response and append it to the dataframe
        if 'Results' in response and 'series' in response['Results']:
            for series_data in response['Results']['series']:
                series_id = series_data['seriesID']
                if series_id in series_dict:
                    county_name = series_dict[series_id]
                    county_data = {f"{i['year']}-{i['period'][1:]}-01": float(i['value']) if 'value' in i else None for i in series_data['data']}
                    temp_df = pd.DataFrame(index=pd.to_datetime(list(county_data.keys())))
                    temp_df[county_name] = pd.Series(list(county_data.values()), index=temp_df.index)

                    # Concatenate the dataframe to store all of the different years we're testing

                    ## TODO:  Make sure this fixes data pull concatenation problem
                    df = pd.concat([df, temp_df], axis=1)
            list_df.append(df)

    df_final = pd.concat(list_df)

    return df_final


# Now, it should return a single dataframe for a single subsector.

# This frame should contain all of the counties, with dates from 2000-2024.

# Additionally, if a county doesn't have data for a specific month, that cell should be empty

# bls_query_update(series_dict=series_dict, dates = (2022, 2024), api_key = dict_api[user])

In [ ]:
# Now let's run the final function

# Should also try and make it so that you can choose what counties you want.

# This will be done when we make the function in tandem with excel

def full_bls(sector_list, df, dates, key, pre, data_type):

    """
    This function combines the dict_maker() and bls_query_update() to make a set of Series IDs for multiple MSAs, sectors, and year range, all
    defined by user. We then return the data in a set of dfs, separated by industry in the sector list. Meaning, df[0] will contain information
    only for the first sector in the sector list. 
    """
    
    # Initialize an empty list so we can iterate over multiple dictionaries
    sector_chamber = []

    # Initialize empty list for each dataframe we will end up making
    df_chamber = []

    # Loop over each sector we want to test
    for i in sector_list:
        sector_chamber.append(dict_maker(df, i, pre, data_type))

    # Now with the sector_holders list containing each set of series we want, we can run our query function iteratively
    
    # Iteratively make each dataframe
    for sector_dict in sector_chamber:
        df_chamber.append(bls_query_update(sector_dict, dates, api_key = key))

    return(df_chamber)
# Perhaps we can multiprocess this. For loops very bad for efficiency, and this is going to be a p lengthy process. 


***

## Pull Data

***

In [ ]:
# Reading in MSA inputs
df_params = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'BLS_MSA')
indicator_name = df_params['indicator_name'].values[0]
year_start     = int(df_params['year_start'    ].values[0])
year_end       = int(df_params['year_end'      ].values[0])


df_peer_msa = df_params[['msa', 'msa_label']]

df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object})
df_peer_msa = df_peer_msa.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
df_peer_msa = df_peer_msa.drop('MSA_ID', axis = 1)

print(indicator_name)
print(year_start    )
print(year_end      )
df_peer_msa.head()

In [ ]:
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_industries.head()

In [ ]:
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]
df_datatypes.head()

In [ ]:
# Subset
df_peer_msa_in = df_peer_msa[df_peer_msa['msa_label'].str.contains('Sac|Austin')]
df_peer_msa_in

In [ ]:
# Testing Agg = Total Nonfarm - Total Priv
# sectors = ['00000000', '05000000', '08000000']

# Inputs
sectors = list(df_industries['industry_code'].values)
year_start = int(df_params['year_start'].values[0])
year_end   = int(df_params['year_end'  ].values[0])
data_type  = df_datatypes['data_type_code'].values[0]


dfs = full_bls(key = dict_api[user]
               , sector_list = sectors
               , df = df_peer_msa_in
               , dates = (year_start, year_end)
               , pre = "SMU"
               , data_type = data_type)

In [ ]:
dfs[0]

In [ ]:
dfs_test = dfs.copy()

for ii in range(len(dfs_test)):
    df_temp = dfs_test[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date')
    df_temp = pd.melt(df_temp
               , id_vars = 'date'
               , var_name = 'MSA'
               , value_name = 'Total Jobs')
    dfs_test[ii] = df_temp

dfs_test[0]

In [ ]:
list(df_industries['Variable'].values)

In [ ]:
df_private    = dfs_test[0]
df_government = dfs_test[1]

In [ ]:
df_private.head()

In [ ]:
df_government.head()

Code graveyard

***

### **Testing Blocks (API example from online source)**


***

In [ ]:
# # Series stored as a dictionary (unemployment rate by ethnicity)
# series_dict = {
#     'LNS14000003': 'White',
#     'LNS14000006': 'Black',
#     'LNS14000009': 'Hispanic'}

# # Start year and end year
# dates = ('2008', '2017')

In [ ]:
# # Specify json as content type to return
# headers = {'Content-type': 'application/json'}

# # Submit the list of series as data
# data = json.dumps({
#     "seriesid": list(series_dict.keys()),
#     "startyear": dates[0],
#     "endyear": dates[1]})

# # Post request for the data
# p = requests.post(
#     '{}{}'.format(url, key),
#     headers=headers,
#     data=data).json()['Results']['series']

In [ ]:
# # Date index from first series
# date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# # Empty dataframe to fill with values
# df = pd.DataFrame()

# # Build a pandas series from the API results, p
# for s in p:
#     df[series_dict[s['seriesID']]] = pd.Series(
#         index = pd.to_datetime(date_list),
#         data = [i['value'] for i in s['data']]
#         ).astype(float).iloc[::-1]

# # Show last 5 results
# df.tail()

In [ ]:
# # Series stored as a dictionary
# series_dict = {
#     # 'LNS12000000': 'Agricultural Total Employment'
#     'SMU48124202023800001': 'Agricultural Total Employment1'
#     , 'SMU06409002023800001': 'Agricultural Total Employment2'
# }

# # Start year and end year
# dates = ('2022', '2024')

# # Specify json as content type to return
# headers = {'Content-type': 'application/json'}

# # Submit the list of series as data
# data = json.dumps({
#     "seriesid" : list(series_dict.keys()),
#     "startyear": dates[0],
#     "endyear"  : dates[1]
# })

# # Post request for the data
# p = requests.post(
#     '{}{}'.format(url, key),
#     headers=headers,
#     data=data).json()['Results']['series']
# # Date index from first series
# date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# # Empty dataframe to fill with values
# df = pd.DataFrame()

# # Build a pandas series from the API results, p
# for s in p:
#     df[series_dict[s['seriesID']]] = pd.Series(
#         index = pd.to_datetime(date_list),
#         data = [i['value'] for i in s['data']]
#         ).astype(float).iloc[::-1]

# # Show last 5 results
# df.tail()
